In [2]:
from pyspark.sql import SparkSession

In [10]:
spark = SparkSession.builder \
    .appName("Case Study") \
    .getOrCreate()

In [3]:
spark = SparkSession.builder \
    .appName("Case Study") \
    .getOrCreate()

26/06/15 04:47:21 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [11]:
customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
returns = spark.read.csv("data/returns.csv", header=True, inferSchema=True)


In [5]:
total_customers = customers.count()
total_products = products.count()
total_orders = orders.count()
total_returned_orders = returns.count()

In [6]:
print(f"Total Customers: {total_customers}")
print(f"Total Products: {total_products}")
print(f"Total Orders: {total_orders}")
print(f"Total Returned Orders: {total_returned_orders}")

Total Customers: 100000
Total Products: 50000
Total Orders: 1000000
Total Returned Orders: 100000


In [6]:
from pyspark.sql.functions import sum, col
order_items = spark.read.csv("data/order_items.csv", header=True, inferSchema=True)
sales_data = order_items.join(products, "product_id")

sales_data.groupBy("category") \
    .agg(
        sum(col("selling_price") * col("quantity"))
        .alias("total_sales")
    ) \
    .show()

[Stage 19:=============================>                            (1 + 1) / 2]

+--------------+-------------------+
|      category|        total_sales|
+--------------+-------------------+
|Home & Kitchen|7.581388732799902E8|
|        Sports|7.433388681300008E8|
|   Electronics|7.442665041099958E8|
|      Clothing|7.419227945699946E8|
|         Books|7.464907783499908E8|
|        Beauty|7.626693058999963E8|
|          Toys|7.446190722999846E8|
+--------------+-------------------+



In [7]:
from pyspark.sql.functions import sum, col

top_customers = orders.join(order_items, "order_id") \
    .groupBy("customer_id") \
    .agg(
        sum(col("selling_price") * col("quantity"))
        .alias("total_purchase")
    ) \
    .orderBy(col("total_purchase").desc()) \
    .limit(10)

top_customers.show()

[Stage 26:>                                                         (0 + 2) / 2]

+-----------+------------------+
|customer_id|    total_purchase|
+-----------+------------------+
|      93094|         181569.68|
|      64560|169060.39999999997|
|      23289|161573.80000000002|
|      52275|153364.78999999998|
|      61218|         153067.55|
|      52034|         152680.05|
|      40442|         151037.32|
|      60528|         148691.95|
|      84830|         148363.84|
|      82593|148281.03999999998|
+-----------+------------------+



In [12]:
from pyspark.sql.functions import year, month, max, sum, col

In [13]:
last_year = orders.select(
    max(year("order_date"))
).collect()[0][0]

print(last_year)

[Stage 48:=============================>                            (1 + 1) / 2]

2024


In [14]:
orders_last_year = orders.filter(
    year("order_date") == last_year
)

In [15]:
sales_df = orders_last_year.join(
    order_items,
    "order_id"
)

In [16]:
monthly_sales = sales_df.groupBy(
    month("order_date").alias("month")
).agg(
    sum(
        col("selling_price") *
        col("quantity")
    ).alias("total_sales")
).orderBy("month")

monthly_sales.show()

[Stage 55:=============================>                            (1 + 1) / 2]

+-----+--------------------+
|month|         total_sales|
+-----+--------------------+
|    1| 4.445777757600032E8|
|    2| 4.153661441999996E8|
|    3| 4.436282454099993E8|
|    4|4.2782097433999574E8|
|    5|4.4481061894999886E8|
|    6| 4.317051540600023E8|
|    7| 4.436705191199994E8|
|    8|4.4109517702000153E8|
|    9| 4.310715260799986E8|
|   10|4.4136378931000113E8|
|   11| 4.336233640400014E8|
|   12| 4.427129083499967E8|
+-----+--------------------+



In [17]:
df = order_items.join(
    products,
    "product_id"
).join(
    orders,
    "order_id"
)

In [18]:
total_orders = df.groupBy(
    "category"
).count()

In [19]:
returned_orders = df.filter(
    col("order_status") == "Returned"
).groupBy(
    "category"
).count()

In [21]:
return_percentage = returned_orders.alias("r").join(
    total_orders.alias("t"),
    "category"
).select(
    "category",
    (
        col("r.count") /
        col("t.count") * 100
    ).alias("return_percentage")
)

return_percentage.show()

+--------------+------------------+
|      category| return_percentage|
+--------------+------------------+
|Home & Kitchen|16.696387840583917|
|        Sports| 16.62794642941293|
|   Electronics|16.580338862069613|
|      Clothing| 16.62882038881496|
|         Books|16.645359482633474|
|        Beauty|16.690396170452935|
|          Toys|16.817837543968885|
+--------------+------------------+



In [23]:
from pyspark.sql.functions import year, month, max, sum, col

In [ ]:
df